In [3]:
!pip install yfinance

In [4]:
# ============================================
# Cell 1: 导入库 + 拉取真实BTC数据
# ============================================

!pip install yfinance -q

import requests
import json
import pandas as pd
import numpy as np
from datetime import datetime
import yfinance as yf

print("=" * 60)
print("📊 拉取真实BTC数据 (Fetching real BTC data)...")
print("=" * 60)

# 拉BTC-USD最近30天的1小时K线
btc = yf.Ticker("BTC-USD")
df = btc.history(period="30d", interval="1h")

print(f"✅ 拉到 {len(df)} 根1小时K线")
print(f"   时间范围: {df.index[0]} → {df.index[-1]}")
print(f"   价格范围: ${df['Close'].min():.0f} ~ ${df['Close'].max():.0f}")
print(f"   最新价格: ${df['Close'].iloc[-1]:.0f}")

📊 拉取真实BTC数据 (Fetching real BTC data)...
✅ 拉到 705 根1小时K线
   时间范围: 2026-06-12 00:00:00+00:00 → 2026-07-11 08:00:00+00:00
   价格范围: $58052 ~ $67242
   最新价格: $64186


In [5]:
# ============================================
# Cell 2: 计算做市商指标 (Calculating MM metrics)
# ============================================

print("=" * 60)
print("📈 计算做市商指标 (Calculating MM metrics)...")
print("=" * 60)

# --- 三行公式①：波动率 (Volatility) ---
df['returns'] = df['Close'].pct_change()
volatility = df['returns'].std()
print(f"📊 小时波动率: {volatility:.4%}")

# --- 三行公式②：动态价差 (Dynamic Spread) ---
base_spread = 20  # 基础价差 $20
volatility_multiplier = volatility * 100  # 波动率越大，价差越宽
dynamic_spread = base_spread * (1 + volatility_multiplier)
print(f"📊 动态价差: ${dynamic_spread:.1f} (基础${base_spread} + 波动率调整)")

# --- 三行公式③：ATR (Average True Range / 平均真实波幅) ---
df['HL_range'] = df['High'] - df['Low']
atr = df['HL_range'].rolling(14).mean()
print(f"📊 ATR(14): ${atr.iloc[-1]:.0f}")

📈 计算做市商指标 (Calculating MM metrics)...
📊 小时波动率: 0.4872%
📊 动态价差: $29.7 (基础$20 + 波动率调整)
📊 ATR(14): $182


In [6]:
# ============================================
# Cell 3: 回测做市策略 (Backtesting MM strategy)
# ============================================

print("=" * 60)
print("🔄 回测做市策略 (Backtesting MM strategy)...")
print("=" * 60)

trades = []
position = 0
cash = 0
spread = dynamic_spread  # 用动态价差

# 从第15根K线开始（给ATR留计算窗口）
for i in range(15, len(df)):
    mid_price = df['Close'].iloc[i]
    bid_price = mid_price - spread / 2
    ask_price = mid_price + spread / 2

    # 模拟：50%概率双边成交
    if np.random.rand() < 0.3:  # 30%成交率（更保守）
        size = 0.01  # 每次0.01 BTC

        # --- 三行公式④：单笔收益 (Round Trip PnL) ---
        pnl = (ask_price - bid_price) * size
        trades.append({
            'time': df.index[i],
            'mid_price': mid_price,
            'bid': bid_price,
            'ask': ask_price,
            'pnl': pnl
        })
        cash += pnl

print(f"✅ 完成 {len(trades)} 笔交易")

🔄 回测做市策略 (Backtesting MM strategy)...
✅ 完成 205 笔交易


In [11]:
# ============================================
# Cell 4: 统计回测结果 (Calculate backtest statistics)
# ============================================

print("=" * 60)
print("📊 回测统计 (Backtest Statistics)...")
print("=" * 60)

df_trades = pd.DataFrame(trades)

# --- 三行公式⑤：总收益 (Total PnL) ---
total_pnl = df_trades['pnl'].sum()
print(f"💰 总收益: ${total_pnl:.2f}")

# --- 三行公式⑥：胜率 (Win Rate) ---
win_rate = (df_trades['pnl'] > 0).mean()
print(f"🎯 胜率: {win_rate:.1%}")

# --- 三行公式⑦：最大回撤 (Max Drawdown) ---
cumulative = df_trades['pnl'].cumsum()
max_dd = cumulative.cummin().min()
print(f"📉 最大回撤: ${max_dd:.2f}")

# --- 三行公式⑧：夏普比率 (Sharpe Ratio) ---
avg_return = df_trades['pnl'].mean()
std_return = df_trades['pnl'].std()
sharpe = (avg_return / std_return) * np.sqrt(8760) if std_return > 0 else 0  # 8760=一年小时数
print(f"⚡ 夏普比率: {sharpe:.2f}")

📊 回测统计 (Backtest Statistics)...
💰 总收益: $60.98
🎯 胜率: 100.0%
📉 最大回撤: $0.30
⚡ 夏普比率: 711160787659250.00


In [12]:
# ============================================
# Cell 5: 本地智能分析（不调API）
# ============================================

print("=" * 60)
print("🤖 本地策略分析 (Local Strategy Analysis)...")
print("=" * 60)

def analyze_strategy(volatility, dynamic_spread, atr, total_pnl, win_rate, max_dd, sharpe, num_trades):
    """根据回测数据自动给出优化建议"""

    print("\n📋 回测数据汇总：")
    print(f"   波动率: {volatility:.4%}")
    print(f"   动态价差: ${dynamic_spread:.1f}")
    print(f"   ATR(14): ${atr:.0f}")
    print(f"   交易次数: {num_trades}")
    print(f"   总收益: ${total_pnl:.2f}")
    print(f"   胜率: {win_rate:.1%}")
    print(f"   最大回撤: ${max_dd:.2f}")
    print(f"   夏普比率: {sharpe:.2f}")

    print("\n" + "-" * 40)
    print("🤖 AI优化建议：")
    print("-" * 40)

    # 建议1：价差问题
    if win_rate > 0.95:
        print("1️⃣ 价差过宽，胜率100%但交易频率太低。")
        print("   建议：缩小价差到ATR的10-20%，提升成交频率。")
    elif win_rate < 0.5:
        print("1️⃣ 价差过窄，亏损交易太多。")
        print("   建议：扩大价差，覆盖更多波动风险。")
    else:
        print("1️⃣ 价差设置合理，胜率在正常范围。")

    # 建议2：价差vs ATR
    spread_atr_ratio = dynamic_spread / atr if atr > 0 else 0
    print(f"\n2️⃣ 价差/ATR比率: {spread_atr_ratio:.1%}")
    if spread_atr_ratio < 0.1:
        print("   价差远小于ATR，容易被大波动吃掉。建议扩大。")
    elif spread_atr_ratio > 0.5:
        print("   价差远大于ATR，成交率会很低。建议缩小。")
    else:
        print("   比率合理，在0.1-0.5之间。")

    # 建议3：库存管理
    print("\n3️⃣ 库存管理建议：")
    print("   当前策略没有库存偏移（Inventory Skew）。")
    print("   建议：引入库存偏斜机制——")
    print("   - 持仓多时，买一价下调，卖一价下调（促进卖出）")
    print("   - 持仓空时，买一价上调，卖一价上调（促进买入）")
    print("   这样可以控制风险敞口，避免单边持仓过大。")

# 调用分析函数
analyze_strategy(
    volatility=volatility,
    dynamic_spread=dynamic_spread,
    atr=atr.iloc[-1],
    total_pnl=total_pnl,
    win_rate=win_rate,
    max_dd=max_dd,
    sharpe=sharpe,
    num_trades=len(trades)
)

🤖 本地策略分析 (Local Strategy Analysis)...

📋 回测数据汇总：
   波动率: 0.4872%
   动态价差: $29.7
   ATR(14): $182
   交易次数: 205
   总收益: $60.98
   胜率: 100.0%
   最大回撤: $0.30
   夏普比率: 711160787659250.00

----------------------------------------
🤖 AI优化建议：
----------------------------------------
1️⃣ 价差过宽，胜率100%但交易频率太低。
   建议：缩小价差到ATR的10-20%，提升成交频率。

2️⃣ 价差/ATR比率: 16.4%
   比率合理，在0.1-0.5之间。

3️⃣ 库存管理建议：
   当前策略没有库存偏移（Inventory Skew）。
   建议：引入库存偏斜机制——
   - 持仓多时，买一价下调，卖一价下调（促进卖出）
   - 持仓空时，买一价上调，卖一价上调（促进买入）
   这样可以控制风险敞口，避免单边持仓过大。


In [13]:
# ============================================
# Cell 6: 输出交易记录前5笔
# ============================================

print("=" * 60)
print("📝 前5笔交易记录 (First 5 trades):")
print("=" * 60)
print(df_trades.head().to_string(index=False))

print("\n" + "=" * 60)
print("✅ 回测完成！(Backtest Complete!)")
print("=" * 60)
print(f"总结：用${dynamic_spread:.1f}动态价差做BTC做市，")
print(f"      30天产生${total_pnl:.2f}收益，胜率{win_rate:.0%}")
print("=" * 60)

📝 前5笔交易记录 (First 5 trades):
                     time    mid_price          bid          ask      pnl
2026-06-12 21:00:00+00:00 63435.609375 63420.737251 63450.481499 0.297442
2026-06-13 02:00:00+00:00 63605.261719 63590.389595 63620.133843 0.297442
2026-06-13 03:00:00+00:00 63494.140625 63479.268501 63509.012749 0.297442
2026-06-13 04:00:00+00:00 63472.429688 63457.557564 63487.301811 0.297442
2026-06-13 06:00:00+00:00 63694.859375 63679.987251 63709.731499 0.297442

✅ 回测完成！(Backtest Complete!)
总结：用$29.7动态价差做BTC做市，
      30天产生$60.98收益，胜率100%
